In [1]:
#Importing Libraries
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [2]:
print(cv2.__version__)
print(cv2.__file__)
print(hasattr(cv2, 'CascadeClassifier'))

4.10.0
C:\Users\Admin\anaconda3\Lib\site-packages\cv2\__init__.py
True


In [3]:
# Define paths and hyper-parameters
data_dir = r"C:\Users\Admin\OneDrive\Desktop\AI\facemask_dataset"
batch_size = 32
image_size = (256, 256)

In [4]:
#Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=1/3,
    subset="training",
    seed=0,
    image_size=image_size,
    batch_size=batch_size
)
#Load testing dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=1/3,
    subset="validation",
    seed=0,
    image_size=image_size,
    batch_size=batch_size
)

Found 3846 files belonging to 2 classes.
Using 2564 files for training.
Found 3846 files belonging to 2 classes.
Using 1282 files for validation.


In [5]:
#Splitting the images into y_train, y_test

x_train = []
y_train = []

for images, labels in train_ds.unbatch():
    x_train.append(images.numpy())
    y_train.append(labels.numpy())


#Splitting the images into y_train, y_test
x_test = []
y_test = []

for images, labels in test_ds.unbatch():
    x_test.append(images.numpy())
    y_test.append(labels.numpy())


In [6]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

In [7]:
print(x_train.shape)
print(x_test.shape)

(2564, 256, 256, 3)
(1282, 256, 256, 3)


In [8]:
x_train = x_train.reshape(x_train.shape[0], 256, 256, 3)
x_test = x_test.reshape(x_test.shape[0], 256, 256, 3)

In [9]:
input_shape = (256,256,3)

In [10]:
#Convert datatype to float32
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

In [11]:
#Normalize pixel value from (0-255) to (0-1)
# x_train /= 255
# x_test /= 255

In [12]:
#Importing Libraries for training
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [13]:
# Load MobileNetV2 base model
baseModel = MobileNetV2(
    weights="imagenet", 
    include_top=False, 
    input_tensor=Input(shape=input_shape)
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19744\3851346040.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  baseModel = MobileNetV2(


In [14]:
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

x_train = preprocess_input(x_train)
x_test = preprocess_input(x_test)

In [15]:
# Construct head of the model
headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(128, activation="relu")(headModel)
#headModel = Dropout(0.5)(headModel)
headModel = Dense(2, activation="softmax")(headModel) # 2 classes: with_mask, without_mask

model = Model(inputs=baseModel.input, outputs=headModel)

In [16]:
# Freeze base model layers
for layer in baseModel.layers:
    layer.trainable = False

In [17]:
# Compile 
model.compile(loss="sparse_categorical_crossentropy", optimizer=Adam(learning_rate=1e-4), metrics=["accuracy"])

In [18]:
#Train
model.fit(x=x_train, y=y_train, epochs=10)

Epoch 1/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 70s 668ms/step - accuracy: 0.9353 - loss: 0.1797
Epoch 2/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 54s 667ms/step - accuracy: 0.9883 - loss: 0.0495
Epoch 3/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 55s 676ms/step - accuracy: 0.9922 - loss: 0.0327
Epoch 4/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 57s 697ms/step - accuracy: 0.9945 - loss: 0.0236
Epoch 5/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 55s 677ms/step - accuracy: 0.9969 - loss: 0.0177
Epoch 6/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 56s 694ms/step - accuracy: 0.9980 - loss: 0.0144
Epoch 7/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 57s 698ms/step - accuracy: 0.9980 - loss: 0.0116
Epoch 8/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 54s 668ms/step - accuracy: 0.9984 - loss: 0.0095
Epoch 9/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 54s 665ms/step - accuracy: 0.9988 - loss: 0.0078
Epoch 10/10
81/81 ━━━━━━━━━━━━━━━━━━━━ 54s 666ms/step - accuracy: 0.9988 - loss: 0.0067


In [19]:
model.evaluate(x_test, y_test)

41/41 ━━━━━━━━━━━━━━━━━━━━ 31s 660ms/step - accuracy: 0.9938 - loss: 0.0210


[0.02102033793926239, 0.9937597513198853]

In [20]:

class_names = train_ds.class_names
print(class_names)

['with_mask', 'without_mask']


In [21]:
# import cv2
# print("OpenCV version:", cv2.__version__)
# print("OpenCV location:", cv2.__file__)
# print("CascadeClassifier available:",
#       hasattr(cv2, 'CascadeClassifier'))

In [22]:
# Load face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

In [23]:
# Open laptop camera
cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        print("Unable to access camera")
        break

    # Convert frame to grayscale for face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:

        # Extract face
        face = frame[y:y+h, x:x+w]

        # Convert BGR to RGB
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

        # Resize image to 256 x 256
        face = cv2.resize(face, (256, 256))

        # Convert to float32
        face = face.astype("float32")

        # Preprocess image
        face = preprocess_input(face)

        # Add batch dimension
        face = np.expand_dims(face, axis=0)

        # Predict
        prediction = model.predict(face, verbose=0)

        # Get class index
        class_index = np.argmax(prediction[0])

        # Get class name
        label = class_names[class_index]

        # Get confidence
        confidence = prediction[0][class_index] * 100

        # Draw rectangle and prediction
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"{label}: {confidence:.2f}%",
            (x, y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    # Display camera output
    cv2.imshow("Face Mask Detection", frame)

    # Press q to close camera
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release camera
cap.release()
cv2.destroyAllWindows()

In [24]:
# Save the trained CNN model
model.save("facemask.keras")


In [25]:
from tensorflow.keras.models import load_model
model = load_model("facemask.keras")

In [26]:
import pickle
filename = "facemask.sav"
pickle.dump(model, open(filename, "wb"))